In [ ]:
XENO_CANTO_API_KEY = "<INSERT_API_KEY>"

In [ ]:
from pathlib import Path
import os
import time
import urllib.request
import requests

import pandas as pd
import numpy as np

import librosa
import birdnet_analyzer

import soundfile as sf


from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GroupShuffleSplit

from joblib import dump, load

PROJECT_ROOT = Path.cwd()

## Change to Audio Directory
EINDHOVEN_DIR = PROJECT_ROOT / "ZooEindhoven_April2025" / "fl_zoo_eindhoven_20250426"

METADATA_PATH = EINDHOVEN_DIR / "fl_zoo_eindhoven_20250426_meta.xlsx"

AUDIO_ROOT = EINDHOVEN_DIR.parent

PROCESSED_DIR = PROJECT_ROOT / "processed_data"
XENO_DIR = PROJECT_ROOT / "xeno_canto_data"
FEATURE_DIR = PROCESSED_DIR / "call_type_features"
MODEL_DIR = PROCESSED_DIR / "models"

for folder in [PROCESSED_DIR, XENO_DIR, FEATURE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root exists:", PROJECT_ROOT.exists())
print("Eindhoven dir exists:", EINDHOVEN_DIR.exists())
print("Metadata exists:", METADATA_PATH.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

Project root exists: True
Eindhoven dir exists: False
Metadata exists: False
Audio root exists: False


In [68]:
metadata_df = pd.read_excel(METADATA_PATH)

metadata_df["metadata_row_id"] = metadata_df.index

def build_full_audio_path(relative_filename):
    if pd.isna(relative_filename):
        return None
    
    relative_filename = str(relative_filename).replace("\\", "/")
    return AUDIO_ROOT / relative_filename

metadata_df["audio_path"] = metadata_df["filename"].apply(build_full_audio_path)

metadata_df["audio_exists"] = metadata_df["audio_path"].apply(
    lambda p: Path(p).exists() if p is not None else False
)

print("Metadata shape:", metadata_df.shape)
print("Audio exists:")
print(metadata_df["audio_exists"].value_counts())
print(metadata_df.columns.tolist())

metadata_df[["metadata_row_id", "filename", "audio_path", "audio_exists"]].head()

Metadata shape: (97864, 33)
Audio exists:
audio_exists
True     82988
False    14876
Name: count, dtype: int64
['Unnamed: 0.1', 'Unnamed: 0', 'sessionId', 'time', 'filename', 'th1', 'th1_value', 'th2', 'th2_value', 'th3', 'th3_value', 'wudate', 'datetime', 'precipRate', 'pressureMax', 'dewptAvg', 'windgustHigh', 'windspeedAvg', 'tempAve', 'humidityAvg', 'winddirAvg', 'uvHigh', 'solarRadiationHigh', 'lon', 'lat', 'MIT_AST_label', 'perch_prediction', 'birdnet_prediction', 'overlap', 'fusion_model_prediction', 'metadata_row_id', 'audio_path', 'audio_exists']


,metadata_row_id,filename,audio_path,audio_exists
0,0,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,False
1,1,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,False
2,2,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,False
3,3,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,False
4,4,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,False


In [ ]:
zoo_segments_df = metadata_df[metadata_df["audio_exists"]].copy()

zoo_segments_df["segment_path"] = zoo_segments_df["audio_path"].astype(str)
zoo_segments_df["segment_file_name"] = zoo_segments_df["audio_path"].apply(lambda p: Path(p).name)
zoo_segments_df["original_file_name"] = zoo_segments_df["segment_file_name"]
zoo_segments_df["original_path"] = zoo_segments_df["segment_path"]
zoo_segments_df["segment_index"] = 0
zoo_segments_df["start_sec"] = 0.0
zoo_segments_df["end_sec"] = None

print("Usable audio rows:", len(zoo_segments_df))

zoo_segments_df.to_csv(PROCESSED_DIR / "zoo_eindhoven_april2025_segments.csv", index=False)

zoo_segments_df[[
    "metadata_row_id",
    "filename",
    "segment_path",
    "datetime",
    "MIT_AST_label",
    "birdnet_prediction",
    "fusion_model_prediction"
]].head()

Usable audio rows: 82988


,metadata_row_id,filename,segment_path,datetime,MIT_AST_label,birdnet_prediction,fusion_model_prediction
39,39,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,2025-04-26 11:38:04,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
40,40,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,2025-04-26 11:38:09,Fowl,Egretta garzetta Little Egret (0.9274),Egretta garzetta Little Egret (0.9275)\nPasser...
41,41,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,2025-04-26 11:38:20,Frog,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
42,42,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,2025-04-26 11:38:23,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997)
43,43,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,2025-04-26 11:38:26,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...


In [ ]:
def has_bird_prediction(row):
    birdnet = str(row.get("birdnet_prediction", "")).strip()
    perch = str(row.get("perch_prediction", "")).strip()
    fusion = str(row.get("fusion_model_prediction", "")).strip()
    
    invalid_values = ["", "nan", "None"]
    
    return (
        birdnet not in invalid_values or
        perch not in invalid_values or
        fusion not in invalid_values
    )

zoo_segments_df["has_bird_prediction"] = zoo_segments_df.apply(has_bird_prediction, axis=1)

print(zoo_segments_df["has_bird_prediction"].value_counts())

zoo_predict_df = zoo_segments_df[zoo_segments_df["has_bird_prediction"]].copy()

print("Rows to classify:", len(zoo_predict_df))

zoo_predict_df[[
    "metadata_row_id",
    "filename",
    "MIT_AST_label",
    "birdnet_prediction",
    "fusion_model_prediction",
    "segment_path"
]].head()

has_bird_prediction
True    82988
Name: count, dtype: int64
Rows to classify: 82988


,metadata_row_id,filename,MIT_AST_label,birdnet_prediction,fusion_model_prediction,segment_path
39,39,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
40,40,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,Fowl,Egretta garzetta Little Egret (0.9274),Egretta garzetta Little Egret (0.9275)\nPasser...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
41,41,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,Frog,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
42,42,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997),D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
43,43,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...


In [ ]:
TARGET_SPECIES = [
    {"english": "House Sparrow", "gen": "Passer", "sp": "domesticus"},

    {"english": "Black Kite", "gen": "Milvus", "sp": "migrans"},
    {"english": "Black Stork", "gen": "Ciconia", "sp": "nigra"},
    {"english": "Chiloe Wigeon", "gen": "Mareca", "sp": "sibilatrix"},
    {"english": "Goliath Heron", "gen": "Ardea", "sp": "goliath"},
    {"english": "Greater Flamingo", "gen": "Phoenicopterus", "sp": "roseus"},
    {"english": "Little Egret", "gen": "Egretta", "sp": "garzetta"},
    {"english": "Northern Bald Ibis", "gen": "Geronticus", "sp": "eremita"},
    {"english": "Pink-backed Pelican", "gen": "Pelecanus", "sp": "rufescens"},
    {"english": "Scarlet Ibis", "gen": "Eudocimus", "sp": "ruber"},
    {"english": "Straw-necked Ibis", "gen": "Threskiornis", "sp": "spinicollis"},
]

In [76]:
def search_xeno_canto(gen, sp, call_type, api_key, max_pages=2, per_page=100):
    url = "https://xeno-canto.org/api/3/recordings"
    
    query = f'gen:{gen} sp:{sp} type:{call_type} grp:birds q:">C"'
    
    headers = {
        "User-Agent": "bird-call-type-classifier/1.0 student-project"
    }
    
    records = []
    
    for page in range(1, max_pages + 1):
        response = requests.get(
            url,
            params={
                "query": query,
                "key": api_key,
                "page": page,
                "per_page": per_page
            },
            headers=headers,
            timeout=30
        )
        
        if response.status_code != 200:
            print("Request failed")
            print("URL:", response.url)
            print("Status:", response.status_code)
            print(response.text[:1000])
            response.raise_for_status()
        
        data = response.json()
        page_records = data.get("recordings", [])
        records.extend(page_records)
        
        print(f"{gen} {sp} {call_type} page {page}: {len(page_records)} recordings")
        
        if page >= int(data.get("numPages", 1)):
            break
        
        time.sleep(1)
    
    return pd.DataFrame(records)

def download_xeno_recordings(df, species_name, label, limit=50):
    out_dir = XENO_DIR / label
    out_dir.mkdir(parents=True, exist_ok=True)
    
    rows = []
    
    subset = df.head(limit)
    
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=f"{species_name} {label}"):
        file_url = row["file"]
        
        if file_url.startswith("//"):
            file_url = "https:" + file_url
        
        xc_id = row["id"]
        out_path = out_dir / f"{species_name.replace(' ', '_')}_XC{xc_id}.mp3"
        
        try:
            urllib.request.urlretrieve(file_url, out_path)
            
            rows.append({
                "xc_id": xc_id,
                "species": species_name,
                "label": label,
                "quality": row.get("q"),
                "type": row.get("type"),
                "path": str(out_path),
                "url": row.get("url"),
                "license": row.get("lic")
            })
            
            time.sleep(0.3)
            
        except Exception as e:
            print(f"Failed download XC{xc_id}: {e}")
    
    return rows

all_xeno_rows = []

for species in TARGET_SPECIES:
    for label in ["song", "call"]:
        df = search_xeno_canto(
            gen=species["gen"],
            sp=species["sp"],
            call_type=label,
            api_key=XENO_CANTO_API_KEY,
            max_pages=2,
            per_page=100
        )
        
        if df.empty:
            print(f"No {label} recordings found for {species['english']}")
            continue
        
        rows = download_xeno_recordings(
            df=df,
            species_name=species["english"],
            label=label,
            limit=50
        )
        
        all_xeno_rows.extend(rows)

xeno_reference_df = pd.DataFrame(all_xeno_rows)
xeno_reference_df.to_csv(PROCESSED_DIR / "xeno_reference_song_call_files.csv", index=False)

print(xeno_reference_df["label"].value_counts())
xeno_reference_df.head()

Passer domesticus song page 1: 100 recordings
Passer domesticus song page 2: 100 recordings


House Sparrow song: 100%|██████████| 50/50 [00:55<00:00,  1.11s/it]


Passer domesticus call page 1: 100 recordings
Passer domesticus call page 2: 100 recordings


House Sparrow call: 100%|██████████| 50/50 [01:48<00:00,  2.17s/it]


Milvus migrans song page 1: 35 recordings


...

Scarlet Ibis call: 100%|██████████| 4/4 [00:02<00:00,  1.89it/s]


Threskiornis spinicollis song page 1: 0 recordings
No song recordings found for Straw-necked Ibis
Threskiornis spinicollis call page 1: 3 recordings


Straw-necked Ibis call: 100%|██████████| 3/3 [00:01<00:00,  1.87it/s]

label
call    265
song    108
Name: count, dtype: int64


,xc_id,species,label,quality,type,path,url,license
0,1142080,House Sparrow,song,A,song,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,https://xeno-canto.org/1142080,https://creativecommons.org/publicdomain/zero/...
1,1142025,House Sparrow,song,A,"call, song",D:\project_23\Bird-Call-Type-Analysis\xeno_can...,https://xeno-canto.org/1142025,https://creativecommons.org/publicdomain/zero/...
2,1142024,House Sparrow,song,A,"call, song",D:\project_23\Bird-Call-Type-Analysis\xeno_can...,https://xeno-canto.org/1142024,https://creativecommons.org/publicdomain/zero/...
3,1142023,House Sparrow,song,A,"call, song",D:\project_23\Bird-Call-Type-Analysis\xeno_can...,https://xeno-canto.org/1142023,https://creativecommons.org/publicdomain/zero/...
4,1142022,House Sparrow,song,A,"call, song",D:\project_23\Bird-Call-Type-Analysis\xeno_can...,https://xeno-canto.org/1142022,https://creativecommons.org/publicdomain/zero/...


In [84]:
XENO_SEGMENT_DIR = PROCESSED_DIR / "xeno_song_call_segments"
XENO_SEGMENT_DIR.mkdir(parents=True, exist_ok=True)

def split_audio_into_3s_chunks(audio_path, out_dir, label, species, sr=22050):
    y, sr = librosa.load(audio_path, sr=sr, mono=True)
    
    chunk_samples = int(3.0 * sr)
    rows = []
    
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for idx, start in enumerate(range(0, len(y), chunk_samples)):
        end = start + chunk_samples
        chunk = y[start:end]
        
        if len(chunk) < int(2.0 * sr):
            continue
        
        out_name = f"{Path(audio_path).stem}_chunk_{idx:04d}.wav"
        out_path = out_dir / out_name
        
        sf.write(out_path, chunk, sr)
        
        rows.append({
            "segment_path": str(out_path),
            "segment_file_name": out_name,
            "original_path": str(audio_path),
            "species": species,
            "label": label,
            "segment_index": idx,
            "start_sec": start / sr,
            "end_sec": min(end, len(y)) / sr
        })
    
    return rows

xeno_segment_rows = []

for _, row in tqdm(xeno_reference_df.iterrows(), total=len(xeno_reference_df)):
    out_dir = XENO_SEGMENT_DIR / row["label"]
    
    try:
        rows = split_audio_into_3s_chunks(
            audio_path=row["path"],
            out_dir=out_dir,
            label=row["label"],
            species=row["species"]
        )
        xeno_segment_rows.extend(rows)
    except Exception as e:
        print("Error segmenting:", row["path"], e)

xeno_segments_df = pd.DataFrame(xeno_segment_rows)
xeno_segments_df.to_csv(PROCESSED_DIR / "xeno_song_call_segments.csv", index=False)

print(xeno_segments_df["label"].value_counts())
xeno_segments_df.head()

100%|██████████| 373/373 [19:45<00:00,  3.18s/it]  


label
call    13200
song     1960
Name: count, dtype: int64


,segment_path,segment_file_name,original_path,species,label,segment_index,start_sec,end_sec
0,D:\project_23\Bird-Call-Type-Analysis\processe...,House_Sparrow_XC1142080_chunk_0000.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,House Sparrow,song,0,0.0,3.0
1,D:\project_23\Bird-Call-Type-Analysis\processe...,House_Sparrow_XC1142080_chunk_0001.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,House Sparrow,song,1,3.0,6.0
2,D:\project_23\Bird-Call-Type-Analysis\processe...,House_Sparrow_XC1142080_chunk_0002.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,House Sparrow,song,2,6.0,9.0
3,D:\project_23\Bird-Call-Type-Analysis\processe...,House_Sparrow_XC1142080_chunk_0003.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,House Sparrow,song,3,9.0,12.0
4,D:\project_23\Bird-Call-Type-Analysis\processe...,House_Sparrow_XC1142080_chunk_0004.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,House Sparrow,song,4,12.0,15.0


In [ ]:
# Build a balanced training set from the segment table.This prevents the model from learning "call" as the default answer.

xeno_segments_df = xeno_segments_df.dropna(subset=["segment_path", "label"]).copy()

print("Original label distribution:")
print(xeno_segments_df["label"].value_counts())

call_df = xeno_segments_df[xeno_segments_df["label"] == "call"].copy()
song_df = xeno_segments_df[xeno_segments_df["label"] == "song"].copy()

min_count = min(len(call_df), len(song_df))

balanced_call_df = call_df.sample(n=min_count, random_state=42)
balanced_song_df = song_df.sample(n=min_count, random_state=42)

balanced_segments_df = (
    pd.concat([balanced_call_df, balanced_song_df], ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("\nBalanced label distribution:")
print(balanced_segments_df["label"].value_counts())

balanced_segments_path = PROCESSED_DIR / "xeno_segments_eindhoven_species_balanced.csv"

balanced_segments_df.to_csv(
    balanced_segments_path,
    index=False
)

print("Saved:", balanced_segments_path)

display(balanced_segments_df.head())

Original label distribution:
label
call    13200
song     1960
Name: count, dtype: int64

Balanced label distribution:
label
call    1960
song    1960
Name: count, dtype: int64
Saved: D:\project_23\Bird-Call-Type-Analysis\processed_data\xeno_segments_eindhoven_species_balanced.csv


,segment_path,segment_file_name,original_path,species,label,segment_index,start_sec,end_sec
0,D:\project_23\Bird-Call-Type-Analysis\processe...,Black_Stork_XC421077_chunk_0013.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,Black Stork,call,13,39.0,42.0
1,D:\project_23\Bird-Call-Type-Analysis\processe...,Black_Kite_XC1084278_chunk_0035.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,Black Kite,call,35,105.0,108.0
2,D:\project_23\Bird-Call-Type-Analysis\processe...,House_Sparrow_XC906531_chunk_0014.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,House Sparrow,song,14,42.0,45.0
3,D:\project_23\Bird-Call-Type-Analysis\processe...,House_Sparrow_XC901057_chunk_0002.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,House Sparrow,song,2,6.0,9.0
4,D:\project_23\Bird-Call-Type-Analysis\processe...,Greater_Flamingo_XC1068287_chunk_0108.wav,D:\project_23\Bird-Call-Type-Analysis\xeno_can...,Greater Flamingo,call,108,324.0,327.0


In [88]:
import shutil

BIRDNET_BALANCED_TRAIN_DIR = PROCESSED_DIR / "xeno_song_call_segments_eindhoven_species_balanced"

if BIRDNET_BALANCED_TRAIN_DIR.exists():
    shutil.rmtree(BIRDNET_BALANCED_TRAIN_DIR)

for label in ["call", "song"]:
    (BIRDNET_BALANCED_TRAIN_DIR / label).mkdir(parents=True, exist_ok=True)

for _, row in tqdm(balanced_segments_df.iterrows(), total=len(balanced_segments_df)):
    src = Path(row["segment_path"])
    label = row["label"]
    
    dest = BIRDNET_BALANCED_TRAIN_DIR / label / src.name
    
    if src.exists():
        shutil.copy2(src, dest)

print("Balanced BirdNET call files:", len(list((BIRDNET_BALANCED_TRAIN_DIR / "call").glob("*.wav"))))
print("Balanced BirdNET song files:", len(list((BIRDNET_BALANCED_TRAIN_DIR / "song").glob("*.wav"))))

100%|██████████| 3920/3920 [00:24<00:00, 156.83it/s]

Balanced BirdNET call files: 1960
Balanced BirdNET song files: 1960


In [90]:
def extract_audio_features(audio_path, sr=22050):
    y, sr = librosa.load(audio_path, sr=sr, mono=True)
    
    target_len = int(3.0 * sr)
    
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y)
    
    features = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        centroid.mean(axis=1),
        bandwidth.mean(axis=1),
        rolloff.mean(axis=1),
        zcr.mean(axis=1),
    ])
    
    return features

In [91]:
X_xeno = []
valid_xeno_rows = []

for _, row in tqdm(balanced_segments_df.iterrows(), total=len(balanced_segments_df), desc="Xeno features"):
    try:
        features = extract_audio_features(row["segment_path"])
        X_xeno.append(features)
        valid_xeno_rows.append(row)
    except Exception as e:
        print("Error:", row["segment_path"], e)

X_xeno = np.vstack(X_xeno)
xeno_feature_index_df = pd.DataFrame(valid_xeno_rows)

np.save(FEATURE_DIR / "xeno_song_call_features.npy", X_xeno)
xeno_feature_index_df.to_csv(FEATURE_DIR / "xeno_song_call_feature_index.csv", index=False)

print(X_xeno.shape)
xeno_feature_index_df["label"].value_counts()

Xeno features: 100%|██████████| 3920/3920 [01:00<00:00, 64.63it/s]


(3920, 44)


label
call    1960
song    1960
Name: count, dtype: int64

In [92]:
X_zoo = []
valid_zoo_rows = []

for _, row in tqdm(zoo_predict_df.iterrows(), total=len(zoo_predict_df), desc="Zoo features"):
    try:
        features = extract_audio_features(row["segment_path"])
        X_zoo.append(features)
        valid_zoo_rows.append(row)
    except Exception as e:
        print("Error:", row["segment_path"], e)

X_zoo = np.vstack(X_zoo)
zoo_feature_index_df = pd.DataFrame(valid_zoo_rows)

np.save(FEATURE_DIR / "zoo_eindhoven_april2025_features.npy", X_zoo)
zoo_feature_index_df.to_csv(FEATURE_DIR / "zoo_eindhoven_april2025_feature_index.csv", index=False)

print(X_zoo.shape)

zoo_feature_index_df[[
    "metadata_row_id",
    "filename",
    "segment_path",
    "MIT_AST_label",
    "birdnet_prediction",
    "fusion_model_prediction"
]].head()

Zoo features: 100%|██████████| 82988/82988 [54:11<00:00, 25.52it/s]  


(82988, 44)


,metadata_row_id,filename,segment_path,MIT_AST_label,birdnet_prediction,fusion_model_prediction
39,39,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
40,40,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,Fowl,Egretta garzetta Little Egret (0.9274),Egretta garzetta Little Egret (0.9275)\nPasser...
41,41,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,Frog,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
42,42,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997)
43,43,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,Animal,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...


In [ ]:
X = np.load(FEATURE_DIR / "xeno_song_call_features.npy")
xeno_index = pd.read_csv(FEATURE_DIR / "xeno_song_call_feature_index.csv")

y = xeno_index["label"].map({
    "call": 0,
    "song": 1
}).values

groups = xeno_index["original_path"].values

print("X shape:", X.shape)
print("Label distribution:")
print(xeno_index["label"].value_counts())

# Group split prevents chunks from the same original recording appearing in both train and test set.
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X[train_idx]
X_test = X[test_idx]
y_train = y[train_idx]
y_test = y[test_idx]

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain label distribution:")
print(pd.Series(y_train).map({0: "call", 1: "song"}).value_counts())

print("\nTest label distribution:")
print(pd.Series(y_test).map({0: "call", 1: "song"}).value_counts())

logreg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ))
])

logreg_model.fit(X_train, y_train)

logreg_pred = logreg_model.predict(X_test)

print("\nLogistic Regression performance:")
print(classification_report(y_test, logreg_pred, target_names=["call", "song"]))
print(confusion_matrix(y_test, logreg_pred))

MODEL_PATH = MODEL_DIR / "song_call_classifier_logreg_eindhoven_species_balanced.joblib"
dump(logreg_model, MODEL_PATH)

print("Saved Logistic Regression model:", MODEL_PATH)

X shape: (3920, 44)
Label distribution:
label
call    1960
song    1960
Name: count, dtype: int64

Train shape: (3302, 44)
Test shape: (618, 44)

Train label distribution:
song    1695
call    1607
Name: count, dtype: int64

Test label distribution:
call    353
song    265
Name: count, dtype: int64

Logistic Regression performance:
              precision    recall  f1-score   support

        call       0.88      0.73      0.80       353
        song       0.70      0.87      0.78       265

    accuracy                           0.79       618
   macro avg       0.79      0.80      0.79       618
weighted avg       0.80      0.79      0.79       618

[[256  97]
 [ 35 230]]
Saved Logistic Regression model: D:\project_23\Bird-Call-Type-Analysis\processed_data\models\song_call_classifier_logreg_eindhoven_species_balanced.joblib


In [97]:
mlp_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=500,
        random_state=42,
        early_stopping=True
    ))
])

mlp_model.fit(X_train, y_train)

mlp_pred = mlp_model.predict(X_test)

print("MLP Classification report:")
print(classification_report(y_test, mlp_pred, target_names=["call", "song"]))

print("MLP Confusion matrix:")
print(confusion_matrix(y_test, mlp_pred))

MLP_MODEL_PATH = MODEL_DIR / "song_call_classifier_mlp.joblib"
dump(mlp_model, MLP_MODEL_PATH)

print("Saved MLP model:", MLP_MODEL_PATH)

MLP Classification report:
              precision    recall  f1-score   support

        call       0.82      0.75      0.78       353
        song       0.70      0.78      0.74       265

    accuracy                           0.76       618
   macro avg       0.76      0.76      0.76       618
weighted avg       0.77      0.76      0.76       618

MLP Confusion matrix:
[[265  88]
 [ 59 206]]
Saved MLP model: D:\project_23\Bird-Call-Type-Analysis\processed_data\models\song_call_classifier_mlp.joblib


In [98]:
model = load(MLP_MODEL_PATH)

X_zoo = np.load(FEATURE_DIR / "zoo_eindhoven_april2025_features.npy")
zoo_index = pd.read_csv(FEATURE_DIR / "zoo_eindhoven_april2025_feature_index.csv")

zoo_pred = model.predict(X_zoo)
zoo_proba = model.predict_proba(X_zoo)

zoo_index["predicted_call_type"] = ["song" if p == 1 else "call" for p in zoo_pred]
zoo_index["prob_call"] = zoo_proba[:, 0]
zoo_index["prob_song"] = zoo_proba[:, 1]

zoo_predictions_path = PROCESSED_DIR / "zoo_eindhoven_april2025_call_type_predictions_mlp.csv"
zoo_index.to_csv(zoo_predictions_path, index=False)

print("Saved predictions:", zoo_predictions_path)
zoo_index[[
    "metadata_row_id",
    "filename",
    "predicted_call_type",
    "prob_call",
    "prob_song",
    "birdnet_prediction",
    "fusion_model_prediction"
]].head(20)

Saved predictions: D:\project_23\Bird-Call-Type-Analysis\processed_data\zoo_eindhoven_april2025_call_type_predictions_mlp.csv


,metadata_row_id,filename,predicted_call_type,prob_call,prob_song,birdnet_prediction,fusion_model_prediction
0,39,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.568004,0.431996,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
1,40,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,song,0.458505,0.541495,Egretta garzetta Little Egret (0.9274),Egretta garzetta Little Egret (0.9275)\nPasser...
2,41,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.900230,0.099770,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
3,42,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.919245,0.080755,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997)
4,43,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.786106,0.213894,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
5,44,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.824968,0.175032,Phoenicopterus roseus Greater Flamingo (0.4175),Greater Flamingo (0.6318)\nPasser domesticus H...
6,47,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.958967,0.041033,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997)
7,48,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.937493,0.062507,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
8,49,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.960039,0.039961,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...
9,50,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,call,0.740080,0.259920,Phoenicopterus roseus Greater Flamingo (0.9999),Phoenicopterus roseus Greater Flamingo (0.9997...


In [99]:
prediction_cols = [
    "metadata_row_id",
    "predicted_call_type",
    "prob_call",
    "prob_song"
]

merged_df = metadata_df.merge(
    zoo_index[prediction_cols],
    on="metadata_row_id",
    how="left"
)

print("Merged shape:", merged_df.shape)
print("Rows with predicted call type:")
print(merged_df["predicted_call_type"].notna().value_counts())

merged_df[[
    "metadata_row_id",
    "filename",
    "MIT_AST_label",
    "birdnet_prediction",
    "fusion_model_prediction",
    "predicted_call_type",
    "prob_call",
    "prob_song"
]].head(30)

Merged shape: (97864, 36)
Rows with predicted call type:
predicted_call_type
True     82988
False    14876
Name: count, dtype: int64


,metadata_row_id,filename,MIT_AST_label,birdnet_prediction,fusion_model_prediction,predicted_call_type,prob_call,prob_song
0,0,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
1,1,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
2,2,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
3,3,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
4,4,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
5,5,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
6,6,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
7,7,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
8,8,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
9,9,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN


In [100]:
output_xlsx = PROCESSED_DIR / "fl_zoo_eindhoven_20250426_meta_with_call_type_mlp.xlsx"
output_csv = PROCESSED_DIR / "fl_zoo_eindhoven_20250426_meta_with_call_type_mlp.csv"

merged_df.to_excel(output_xlsx, index=False)
merged_df.to_csv(output_csv, index=False)

print("Saved:", output_xlsx)
print("Saved:", output_csv)

merged_df[[
    "filename",
    "MIT_AST_label",
    "birdnet_prediction",
    "fusion_model_prediction",
    "predicted_call_type",
    "prob_call",
    "prob_song"
]].head(30)

Saved: D:\project_23\Bird-Call-Type-Analysis\processed_data\fl_zoo_eindhoven_20250426_meta_with_call_type_mlp.xlsx
Saved: D:\project_23\Bird-Call-Type-Analysis\processed_data\fl_zoo_eindhoven_20250426_meta_with_call_type_mlp.csv


,filename,MIT_AST_label,birdnet_prediction,fusion_model_prediction,predicted_call_type,prob_call,prob_song
0,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
1,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
2,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
3,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
4,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
5,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
6,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
7,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
8,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
9,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN


In [101]:
merged_df["predicted_call_type"].value_counts(dropna=False)
merged_df[["prob_call", "prob_song"]].describe()

,prob_call,prob_song
count,82988.000000,82988.000000
mean,0.765093,0.234907
std,0.246438,0.246438
min,0.005034,0.000031
25%,0.629150,0.035438
50%,0.861063,0.138937
75%,0.964562,0.370850
max,0.999969,0.994966


## BirdNET Classifier

In [1]:
import sys
import subprocess

result = subprocess.run(
    [sys.executable, "-m", "birdnet_analyzer.train", "-h"],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stdout[:3000])
print(result.stderr[:3000])

Return code: 0
usage: __main__.py [-h] [--fmin FMIN] [--fmax FMAX]
                   [--audio_speed AUDIO_SPEED] [-t THREADS] [-b BATCH_SIZE]
                   [--overlap OVERLAP] [--test_data TEST_DATA]
                   [--crop_mode {center,first,segments,smart}] [-o OUTPUT]
                   [--epochs EPOCHS] [--val_split VAL_SPLIT]
                   [--learning_rate LEARNING_RATE] [--focal-loss]
                   [--focal-loss-gamma FOCAL_LOSS_GAMMA]
                   [--focal-loss-alpha FOCAL_LOSS_ALPHA]
                   [--hidden_units HIDDEN_UNITS] [--dropout DROPOUT]
                   [--label_smoothing] [--mixup]
                   [--upsampling_ratio UPSAMPLING_RATIO]
                   [--upsampling_mode {repeat,linear,mean,smote}]
                   [--model_format {tflite,raven,both}]
                   [--model_save_mode {replace,append}]
                   [--cache_mode {load,save}] [--cache_file CACHE_FILE]
                   [--autotune] [--autotune_trials AU

In [4]:
BIRDNET_TRAIN_DIR = PROCESSED_DIR / "xeno_song_call_segments_eindhoven_species_balanced"

print("Train dir exists:", BIRDNET_TRAIN_DIR.exists())
print("Call folder exists:", (BIRDNET_TRAIN_DIR / "call").exists())
print("Song folder exists:", (BIRDNET_TRAIN_DIR / "song").exists())

call_files = list((BIRDNET_TRAIN_DIR / "call").rglob("*.wav"))
song_files = list((BIRDNET_TRAIN_DIR / "song").rglob("*.wav"))

print("Call wav files:", len(call_files))
print("Song wav files:", len(song_files))

Train dir exists: True
Call folder exists: True
Song folder exists: True
Call wav files: 1960
Song wav files: 1960


In [ ]:
BIRDNET_MODEL_DIR = PROCESSED_DIR / "birdnet_custom_model"
BIRDNET_MODEL_DIR.mkdir(parents=True, exist_ok=True)

BIRDNET_CLASSIFIER_PATH = BIRDNET_MODEL_DIR / "birdnet_song_call_classifier.tflite"

cmd = [
    sys.executable,
    "-m", "birdnet_analyzer.train",
    str(BIRDNET_TRAIN_DIR),
    "-o", str(BIRDNET_CLASSIFIER_PATH),
    "--epochs", "50",
    "--crop_mode", "center",
    "--model_format", "tflite",
    "-b", "32",
    "-t", "4"
]

print("Running command:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)
print("Classifier exists:", BIRDNET_CLASSIFIER_PATH.exists())

Running command:
d:\project_23\Bird-Call-Type-Analysis\.venv\Scripts\python.exe -m birdnet_analyzer.train D:\project_23\Bird-Call-Type-Analysis\processed_data\xeno_song_call_segments_eindhoven_species_balanced -o D:\project_23\Bird-Call-Type-Analysis\processed_data\birdnet_custom_model\birdnet_song_call_classifier.tflite --epochs 50 --crop_mode center --model_format tflite -b 32 -t 4
STDOUT:
Model found!
Loading training data...
	...train data path: D:\project_23\Bird-Call-Type-Analysis\processed_data\xeno_song_call_segments_eindhoven_species_balanced
	...test data path: None
...Done. Loaded 3920 training samples and 2 labels.
Building model...
...Done.
Training model...
Training on 3136 samples, validating on 784 samples.
Epoch 1/50

 1/98 ━━━━━━━━━━━━━━━━━━━━ 4:42 3s/step - AUPRC: 0.4265 - AUROC: 0.4209 - loss: 1.5903
60/98 ━━━━━━━━━━━━━━━━━━━━ 0s 905us/step - AUPRC: 0.4809 - AUROC: 0.5017 - loss: 1.4845
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - AUPRC: 0.5103 - AUROC: 0.5246 - loss: 1

In [ ]:
BIRDNET_PRED_DIR = PROCESSED_DIR / "birdnet_song_call_predictions"
BIRDNET_PRED_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    "-m", "birdnet_analyzer.analyze",
    str(EINDHOVEN_DIR),
    "-o", str(BIRDNET_PRED_DIR),
    "--classifier", str(BIRDNET_CLASSIFIER_PATH),
    "--rtype", "csv",
    "--top_n", "2",
    "--min_conf", "0.0",
    "--skip_existing_results",
    "--combine_results",
    "-b", "16",
    "-t", "4"
]

print("Running command:")
print(" ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print("STDOUT:")
print(result.stdout[-3000:])

print("STDERR:")
print(result.stderr[-3000:])

print("Return code:", result.returncode)

Running command:
d:\project_23\Bird-Call-Type-Analysis\.venv\Scripts\python.exe -m birdnet_analyzer.analyze D:\ZooEindhoven_April2025\fl_zoo_eindhoven_20250426 -o D:\project_23\Bird-Call-Type-Analysis\processed_data\birdnet_song_call_predictions --classifier D:\project_23\Bird-Call-Type-Analysis\processed_data\birdnet_custom_model\birdnet_song_call_classifier.tflite --rtype csv --top_n 2 --min_conf 0.0 --skip_existing_results --combine_results -b 16 -t 4
STDOUT:
oven_April2025\fl_zoo_eindhoven_20250426\9\er_file_2025_04_26_13_37_33.500000.wav as it has already been analyzed
Skipping D:\ZooEindhoven_April2025\fl_zoo_eindhoven_20250426\9\er_file_2025_04_26_13_37_36.250000.wav as it has already been analyzed
Skipping D:\ZooEindhoven_April2025\fl_zoo_eindhoven_20250426\9\er_file_2025_04_26_13_37_42.250000.wav as it has already been analyzed
Skipping D:\ZooEindhoven_April2025\fl_zoo_eindhoven_20250426\9\er_file_2025_04_26_13_37_45.wav as it has already been analyzed
Skipping D:\ZooEindhoven

In [ ]:
all_csv_files = list(BIRDNET_PRED_DIR.rglob("*.csv"))

prediction_csv_files = [
    f for f in all_csv_files
    if "params" not in f.name.lower()
    and "sample_counts" not in f.name.lower()
]

print("Prediction CSV files:", len(prediction_csv_files))

birdnet_rows = []

required_cols = {"Start (s)", "End (s)", "Scientific name", "Common name", "Confidence", "File"}

for csv_file in prediction_csv_files:
    try:
        df_tmp = pd.read_csv(csv_file)
        
        if required_cols.issubset(set(df_tmp.columns)):
            birdnet_rows.append(df_tmp)
        else:
            print("Skipping non-prediction CSV:", csv_file.name, df_tmp.columns.tolist())
            
    except Exception as e:
        print("Could not read:", csv_file, e)

birdnet_raw_df = pd.concat(birdnet_rows, ignore_index=True)

print("Raw BirdNET rows:", len(birdnet_raw_df))
birdnet_raw_df.head()

Prediction CSV files: 82990
Could not read: D:\project_23\Bird-Call-Type-Analysis\processed_data\birdnet_song_call_predictions\BirdNET_CombinedTable.csv No columns to parse from file
Raw BirdNET rows: 165978


,Start (s),End (s),Scientific name,Common name,Confidence,File
0,0.0,2.75,call,call,0.7399,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
1,0.0,2.75,song,song,0.3507,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
2,0.0,2.75,call,call,0.8611,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
3,0.0,2.75,song,song,0.2344,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...
4,0.0,2.75,call,call,0.6619,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...


In [ ]:
birdnet_raw_df["label"] = birdnet_raw_df["Common name"].astype(str).str.lower().str.strip()

birdnet_raw_df = birdnet_raw_df[birdnet_raw_df["label"].isin(["call", "song"])].copy()

birdnet_probs_df = (
    birdnet_raw_df
    .groupby(["File", "label"])["Confidence"]
    .max()
    .unstack()
    .reset_index()
)

if "call" not in birdnet_probs_df.columns:
    birdnet_probs_df["call"] = np.nan

if "song" not in birdnet_probs_df.columns:
    birdnet_probs_df["song"] = np.nan

birdnet_probs_df = birdnet_probs_df.rename(columns={
    "File": "birdnet_file",
    "call": "prob_call_birdnet",
    "song": "prob_song_birdnet"
})

birdnet_probs_df["predicted_call_type_birdnet"] = np.where(
    birdnet_probs_df["prob_call_birdnet"] >= birdnet_probs_df["prob_song_birdnet"],
    "call",
    "song"
)

birdnet_probs_df.head()

label,birdnet_file,prob_call_birdnet,prob_song_birdnet,predicted_call_type_birdnet
0,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,0.7399,0.3507,call
1,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,0.8611,0.2344,call
2,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,0.6619,0.4461,call
3,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,0.9426,0.1935,call
4,D:\ZooEindhoven_April2025\fl_zoo_eindhoven_202...,0.9683,0.1684,call


In [ ]:
def normalize_path_for_merge(path_value):
    if pd.isna(path_value):
        return None
    
    return str(path_value).replace("\\", "/").lower()


birdnet_probs_df["audio_path_key"] = birdnet_probs_df["birdnet_file"].apply(normalize_path_for_merge)

if "audio_path" not in metadata_df.columns:
    metadata_df["audio_path"] = metadata_df["filename"].apply(build_full_audio_path)

if "metadata_row_id" not in metadata_df.columns:
    metadata_df["metadata_row_id"] = metadata_df.index

metadata_df["audio_path_key"] = metadata_df["audio_path"].apply(normalize_path_for_merge)

print("BirdNET predictions:", len(birdnet_probs_df))
print("Metadata rows:", len(metadata_df))

BirdNET predictions: 82989
Metadata rows: 97864


In [ ]:
birdnet_merge_cols = [
    "audio_path_key",
    "predicted_call_type_birdnet",
    "prob_call_birdnet",
    "prob_song_birdnet",
    "birdnet_file"
]

metadata_with_birdnet_df = metadata_df.merge(
    birdnet_probs_df[birdnet_merge_cols],
    on="audio_path_key",
    how="left"
)

print("Merged rows:", len(metadata_with_birdnet_df))

print("Rows with BirdNET call type:")
print(metadata_with_birdnet_df["predicted_call_type_birdnet"].notna().value_counts())

metadata_with_birdnet_df[[
    "filename",
    "MIT_AST_label",
    "birdnet_prediction",
    "fusion_model_prediction",
    "predicted_call_type_birdnet",
    "prob_call_birdnet",
    "prob_song_birdnet"
]].head(30)

Merged rows: 97864
Rows with BirdNET call type:
predicted_call_type_birdnet
True     82988
False    14876
Name: count, dtype: int64


,filename,MIT_AST_label,birdnet_prediction,fusion_model_prediction,predicted_call_type_birdnet,prob_call_birdnet,prob_song_birdnet
0,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
1,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
2,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
3,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
4,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
5,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
6,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
7,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
8,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN
9,fl_zoo_eindhoven_20250426/0/er_file_2025_04_26...,speech_removed,NaN,NaN,NaN,NaN,NaN


In [ ]:
output_xlsx = PROCESSED_DIR / "fl_zoo_eindhoven_20250426_meta_with_call_type_birdnet.xlsx"
output_csv = PROCESSED_DIR / "fl_zoo_eindhoven_20250426_meta_with_call_type_birdnet.csv"

metadata_with_birdnet_df.to_excel(output_xlsx, index=False)
metadata_with_birdnet_df.to_csv(output_csv, index=False)

print("Saved:", output_xlsx)
print("Saved:", output_csv)

Saved: D:\project_23\Bird-Call-Type-Analysis\processed_data\fl_zoo_eindhoven_20250426_meta_with_call_type_birdnet.xlsx
Saved: D:\project_23\Bird-Call-Type-Analysis\processed_data\fl_zoo_eindhoven_20250426_meta_with_call_type_birdnet.csv


In [ ]:
print("Total rows:", len(metadata_with_birdnet_df))

print("\nBirdNET call type labels:")
print(metadata_with_birdnet_df["predicted_call_type_birdnet"].value_counts(dropna=False))

print("\nRows with BirdNET prediction:")
print(metadata_with_birdnet_df["predicted_call_type_birdnet"].notna().value_counts())

print("\nProbabilities:")
print(metadata_with_birdnet_df[["prob_call_birdnet", "prob_song_birdnet"]].describe())

Total rows: 97864

BirdNET call type labels:
predicted_call_type_birdnet
call    78348
NaN     14876
song     4640
Name: count, dtype: int64

Rows with BirdNET prediction:
predicted_call_type_birdnet
True     82988
False    14876
Name: count, dtype: int64

Probabilities:
       prob_call_birdnet  prob_song_birdnet
count       82988.000000       82988.000000
mean            0.799473           0.291938
std             0.154527           0.148451
min             0.073300           0.019700
25%             0.722600           0.177900
50%             0.843700           0.264700
75%             0.917100           0.381600
max             0.998200           0.926700


In [ ]:
comparison_df = merged_df.merge(
    metadata_with_birdnet_df[[
        "metadata_row_id",
        "predicted_call_type_birdnet",
        "prob_call_birdnet",
        "prob_song_birdnet"
    ]],
    on="metadata_row_id",
    how="left"
)

comparison_df["mlp_confidence"] = comparison_df[["prob_call", "prob_song"]].max(axis=1)
comparison_df["birdnet_confidence"] = comparison_df[["prob_call_birdnet", "prob_song_birdnet"]].max(axis=1)

comparison_df["mlp_birdnet_agree"] = (
    comparison_df["predicted_call_type"] == comparison_df["predicted_call_type_birdnet"]
)

print("Rows:", len(comparison_df))

print("\nMLP labels:")
print(comparison_df["predicted_call_type"].value_counts(dropna=False))

print("\nBirdNET labels:")
print(comparison_df["predicted_call_type_birdnet"].value_counts(dropna=False))

print("\nAgreement:")
print(comparison_df["mlp_birdnet_agree"].value_counts(dropna=False))

Rows: 97864

MLP labels:
predicted_call_type
call    69277
NaN     14876
song    13711
Name: count, dtype: int64

BirdNET labels:
predicted_call_type_birdnet
call    78348
NaN     14876
song     4640
Name: count, dtype: int64

Agreement:
mlp_birdnet_agree
True     69333
False    28531
Name: count, dtype: int64


In [ ]:
valid_comparison_df = comparison_df[
    comparison_df["predicted_call_type"].notna() &
    comparison_df["predicted_call_type_birdnet"].notna()
].copy()

agreement_rate = valid_comparison_df["mlp_birdnet_agree"].mean()

print("Valid comparison rows:", len(valid_comparison_df))
print("Agreement rate:", agreement_rate)

pd.crosstab(
    valid_comparison_df["predicted_call_type"],
    valid_comparison_df["predicted_call_type_birdnet"],
    rownames=["MLP"],
    colnames=["BirdNET"]
)

Valid comparison rows: 82988
Agreement rate: 0.835458138526052


BirdNET,call,song
MLP,,
call,66985,2292
song,11363,2348


In [ ]:
output_comparison_csv = PROCESSED_DIR / "fl_zoo_eindhoven_20250426_meta_with_call_type_mlp_and_birdnet.csv"
output_comparison_xlsx = PROCESSED_DIR / "fl_zoo_eindhoven_20250426_meta_with_call_type_mlp_and_birdnet.xlsx"

comparison_df.to_csv(output_comparison_csv, index=False)
comparison_df.to_excel(output_comparison_xlsx, index=False)

print("Saved:", output_comparison_csv)
print("Saved:", output_comparison_xlsx)

Saved: D:\project_23\Bird-Call-Type-Analysis\processed_data\fl_zoo_eindhoven_20250426_meta_with_call_type_mlp_and_birdnet.csv
Saved: D:\project_23\Bird-Call-Type-Analysis\processed_data\fl_zoo_eindhoven_20250426_meta_with_call_type_mlp_and_birdnet.xlsx
